In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

options = Options()

# --- AJUSTES CRÍTICOS PARA DOCKER ---
options.add_argument("--no-sandbox") # Evita problemas de permisos dentro del contenedor
options.add_argument("--disable-dev-shm-usage") # Usa /tmp en lugar de /dev/shm (evita falta de memoria)
options.add_argument("--disable-extensions") # Desactiva extensiones para ahorrar recursos
options.add_argument("--remote-debugging-port=9222") # Ayuda a estabilizar la conexión del driver

# Si el error persiste, descomenta la siguiente línea para probar en modo headless 
# solo para descartar que el problema sea el Display
# options.add_argument("--headless") 

try:
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("¡Éxito! El navegador se ha iniciado correctamente.")
except Exception as e:
    print(f"Error detectado: {e}")

¡Éxito! El navegador se ha iniciado correctamente.


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import time

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# options.add_argument("--headless") # Descomenta si no necesitas ver la GUI

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [7]:
# Lista de diccionarios con la información base de cada hotel
hoteles_a_explorar = [
    {
        "nombre": "Riu Plaza España",
        "url_base": "https://www.tripadvisor.es/Hotel_Review-g187514-d1192534-Reviews-",
        "url_final": "Hotel_Riu_Plaza_Espana-Madrid.html"
    },
    {
        "nombre": "Hotel Arts Barcelona",
        "url_base": "https://www.tripadvisor.es/Hotel_Review-g187497-d228612-Reviews-",
        "url_final": "Hotel_Arts_Barcelona-Barcelona_Catalonia.html"
    }
    # Puedes seguir agregando más aquí...
]

# Configuración de paginación
PAGINAS_POR_HOTEL = 3  # Ajusta cuántas páginas quieres (cada página tiene 10 reseñas)

In [12]:
# CELDA DE EXTRACCIÓN
import pandas as pd
from selenium.webdriver.common.by import By

# 1. Scroll para asegurar que el contenido dinámico se dispare
driver.execute_script("window.scrollTo(0, 1500);")
time.sleep(3)

# 2. Intentamos capturar los bloques con 3 selectores diferentes (por si uno falla)
# El primero es el más moderno (2026), los otros son respaldos.
bloques = driver.find_elements(By.CSS_SELECTOR, "div[data-test-target='HR_CC_Card']") 
if not bloques:
    bloques = driver.find_elements(By.XPATH, "//div[contains(@class, 'review-container')]")
if not bloques:
    bloques = driver.find_elements(By.CSS_SELECTOR, "div.fIrGe._m")

print(f"Bloques encontrados con nuevos selectores: {len(bloques)}")

datos = []
for r in bloques:
    try:
        # Selector de texto basado en el contenedor de comentario
        texto = r.find_element(By.CSS_SELECTOR, "span.yY6yc, div.biGQs._P.pZpGL.K7.ncS1n, span.exSbi").text
        
        # Rating (buscamos la burbuja)
        rating_raw = r.find_element(By.CSS_SELECTOR, "span.ui_bubble_rating").get_attribute("class")
        rating = int(rating_raw.split("_")[-1]) / 10
        
        # Fecha
        fecha = r.find_element(By.CSS_SELECTOR, "div.tePDR, span.tePDR, div.RpeYy").text
        
        datos.append({"Fecha": fecha, "Rating": rating, "Reseña": texto})
    except:
        continue

df_final = pd.DataFrame(datos)
df_final.head()

Bloques encontrados con nuevos selectores: 0


""


In [11]:
driver.get("https://www.tripadvisor.es/Hotel_Review-g187514-d1192534-Reviews-Hotel_Riu_Plaza_Espana-Madrid.html")
time.sleep(8) # Damos mucho tiempo para asegurar carga

# 1. ¿Existen bloques de reseñas?
bloques = driver.find_elements(By.XPATH, "//div[@data-reviewid]")
print(f"Bloques detectados: {len(bloques)}")

if len(bloques) > 0:
    # 2. Si hay bloques, ¿qué clases tienen dentro?
    # Vamos a imprimir el HTML de un bloque para encontrar los nuevos selectores
    print("Contenido del primer bloque:")
    print(bloques[0].get_attribute('innerHTML')[:500]) 
else:
    print("No se detectó ningún bloque. Es posible que haya un CAPTCHA o que el selector @data-reviewid haya cambiado.")

Bloques detectados: 0
No se detectó ningún bloque. Es posible que haya un CAPTCHA o que el selector @data-reviewid haya cambiado.
